In [ ]:
# Crawl outputs of open-gira TC/grid analysis, specifically:
# power/by_storm_set/{STORM_SET}/disruption/pop_affected_RP/{RETURN_PERIOD}.gpq files
# for each return period, plot a bar chart of calibrated (variable threshold) of 1 in N year
# disruption events by region, by country.

In [ ]:
from collections import defaultdict
import os
import re

import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.transforms import blended_transform_factory
import numpy as np

In [ ]:
# Input data paths
country_thresholds_path = "../data/country_thresholds.csv"
wb_income_classification_path = "../data/wb_income_classification.csv"
targets_path = "/data/tc-grid/open-gira/results/power/targets.geoparquet"
results_dir = "/data/tc-grid/open-gira/results"

# Output paths
plot_dir = "../plots"

# Return periods to process and plot
return_periods = [10, 20, 50, 100, 200, 500]
# Mapping from region display name to region bounding box
regions = {
    "Americas": [-103, 4, -48, 32],
    "Indian Ocean": [20, -35, 95, 27],
    "West Pacific": [101.5, -55, 170, 60],
}
# Mapping from open-gira STORM_SET to display name
atmospheres = {
    "STORM_constant": "2000 STORM",
    "STORM_CNRM-CM6-1-HR": "2050 STORM RCP8.5 CNRM",
    "STORM_CMCC-CM2-VHR4": "2050 STORM RCP8.5 CMCC",
    "STORM_EC-Earth3P-HR": "2050 STORM RCP8.5 EC-Earth",
    "STORM_HadGEM3-GC31-HM": "2050 STORM RCP8.5 HadGEM3",
    "IRIS_PRESENT": "2000 IRIS",
    "IRIS_SSP5-2050": "2050 IRIS SSP5-8.5",
    "CHAZ_SSP-585_GCM-UKESM1-0-LL_epoch-2010": "2010 CHAZ UKESM1",
    "CHAZ_SSP-585_GCM-CESM2_epoch-2010": "2010 CHAZ CESM2",
    "CHAZ_SSP-585_GCM-CNRM-CM6-1_epoch-2010": "2010 CHAZ CNRM",
    "CHAZ_SSP-585_GCM-EC-Earth3_epoch-2010": "2010 CHAZ EC-Earth3",
    "CHAZ_SSP-585_GCM-MIROC6_epoch-2010": "2010 CHAZ MIROC6",
    "CHAZ_SSP-585_GCM-UKESM1-0-LL_epoch-2050": "2050 CHAZ SSP5-8.5 UKESM1",
    "CHAZ_SSP-585_GCM-CESM2_epoch-2050": "2050 CHAZ SSP5-8.5 CESM2",
    "CHAZ_SSP-585_GCM-CNRM-CM6-1_epoch-2050": "2050 CHAZ SSP5-8.5 CNRM",
    "CHAZ_SSP-585_GCM-EC-Earth3_epoch-2050": "2050 CHAZ SSP5-8.5 EC-Earth3",
    "CHAZ_SSP-585_GCM-MIROC6_epoch-2050": "2050 CHAZ SSP5-8.5 MIROC6",
    "emanuel_ssp-585_gcm-cesm2_epoch-2005": "2005 Emanuel CESM2",
    "emanuel_ssp-585_gcm-cnrm6_epoch-2005": "2005 Emanuel CNRM6",
    "emanuel_ssp-585_gcm-ecearth6_epoch-2005": "2005 Emanuel EC-Earth6",
    #"emanuel_ssp-585_gcm-fgoals_epoch-2005": "2005 Emanuel FGOALS",
    #"emanuel_ssp-585_gcm-ipsl6_epoch-2005": "2005 Emanuel IPSL6",
    "emanuel_ssp-585_gcm-miroc6_epoch-2005": "2005 Emanuel MIROC6",
    #"emanuel_ssp-585_gcm-mpi6_epoch-2005": "2005 Emanuel MPI6",
    #"emanuel_ssp-585_gcm-mri6_epoch-2005": "2005 Emanuel MRI6",
    "emanuel_ssp-585_gcm-ukmo6_epoch-2005": "2005 Emanuel UKMO6",
    "emanuel_ssp-585_gcm-cesm2_epoch-2050": "2050 Emanuel CESM2",
    "emanuel_ssp-585_gcm-cnrm6_epoch-2050": "2050 Emanuel CNRM6",
    "emanuel_ssp-585_gcm-ecearth6_epoch-2050": "2050 Emanuel EC-Earth6",
    #"emanuel_ssp-585_gcm-fgoals_epoch-2050": "2050 Emanuel FGOALS",
    #"emanuel_ssp-585_gcm-ipsl6_epoch-2050": "2050 Emanuel IPSL6",
    "emanuel_ssp-585_gcm-miroc6_epoch-2050": "2050 Emanuel MIROC6",
    #"emanuel_ssp-585_gcm-mpi6_epoch-2050": "2050 Emanuel MPI6",
    #"emanuel_ssp-585_gcm-mri6_epoch-2050": "2050 Emanuel MRI6",
    "emanuel_ssp-585_gcm-ukmo6_epoch-2050": "2050 Emanuel UKMO6",
}
# Specify group members (GCM variants) of track sets
gcm_groups = {
    "IRIS baseline": ["2000 IRIS"],
    "IRIS SSP5-8.5 2050": ["2050 IRIS SSP5-8.5"],
    "STORM baseline": ["2000 STORM"],
    "STORM RCP8.5 2050": [
        "2050 STORM RCP8.5 CNRM",
        "2050 STORM RCP8.5 CMCC",
        "2050 STORM RCP8.5 EC-Earth",
        "2050 STORM RCP8.5 HadGEM3"
    ],
    "CHAZ baseline": [
        "2010 CHAZ UKESM1",
        "2010 CHAZ CESM2",
        "2010 CHAZ CNRM",
        "2010 CHAZ EC-Earth3",
        "2010 CHAZ MIROC6",
    ],
    "CHAZ SSP5-8.5 2050": [
        "2050 CHAZ SSP5-8.5 UKESM1",
        "2050 CHAZ SSP5-8.5 CESM2",
        "2050 CHAZ SSP5-8.5 CNRM",
        "2050 CHAZ SSP5-8.5 EC-Earth3",
        "2050 CHAZ SSP5-8.5 MIROC6",
    ],   
    "Emanuel baseline": [
        "2005 Emanuel CESM2",
        "2005 Emanuel CNRM6",
        "2005 Emanuel EC-Earth6",
        #"2005 Emanuel FGOALS",
        #"2005 Emanuel IPSL6",
        "2005 Emanuel MIROC6",
        #"2005 Emanuel MPI6",
        #"2005 Emanuel MRI6",
        "2005 Emanuel UKMO6",
    ],
    "Emanuel SSP5-8.5 2050": [
        "2050 Emanuel CESM2",
        "2050 Emanuel CNRM6",
        "2050 Emanuel EC-Earth6",
        #"2050 Emanuel FGOALS",
        #"2050 Emanuel IPSL6",
        "2050 Emanuel MIROC6",
        #"2050 Emanuel MPI6",
        #"2050 Emanuel MRI6",
        "2050 Emanuel UKMO6",
    ],
}
colours = {
    "IRIS baseline": "darkgreen",
    "IRIS SSP5-8.5 2050": "lightgreen",
    "STORM baseline": "purple",
    "STORM RCP8.5 2050": "plum",
    "CHAZ baseline": "red",
    "CHAZ SSP5-8.5 2050": "pink",
    "Emanuel baseline": "blue",
    "Emanuel SSP5-8.5 2050": "lightblue",
}

In [ ]:
def lookup(df: pd.DataFrame, rows, cols) -> np.ndarray:
    """
    Helper function to lookup values based on list of `rows` and `cols` labels.
    Does this really not exist in pandas already?!

    Args:
        df: Table to lookup from
        rows: Iterable of row indicies to select
        cols: Iterable of column indicies to select (same length as `rows`)

    Returns:
        Elements of `df` that are indexed by `rows` and `cols`.
    """
    return df.to_numpy()[df.index.get_indexer(rows), df.columns.get_indexer(cols)]

def process_disruption_data(
    region: str,
    region_bbox: tuple[float, float, float, float],
    return_period: int,
    atmospheres: dict,
    default_threshold: float,
    thresholds_by_country: pd.DataFrame,
    results_dir: str,
    gcm_groups: dict[str, list[str]],
) -> pd.DataFrame:
    """
    Read in disruption figures from disk
    Select appropriate threshold for each country
    Calculate min, mean and max for any multi-GCM track sets

    Args:
        region: Region name, maps to bounding box in global
        region_bbox: Bounding box for region in decimal degrees: xmin, ymin, xmax, ymax
        return_period: Return period of disruption event in years to lookup
        atmospheres: Mapping from open-gira STORM_SET to human readable name
        default_threshold: Threshold to use if we haven't set via another means
        thresholds_by_country: Threshold values to use per-country
        results_dir: Path to open-gira/results folder
        gcm_groups: Mapping from aggregated scenario label to list of member scenario
            names (as they appear in atmospheres values). For each group, individual
            GCM runs are collapsed into a single row with mean/min/max across models.
            Example:
                {
                    "2010 CHAZ": ["2010 CHAZ UKESM1", "2010 CHAZ CESM2", ...],
                    "2050 CHAZ SSP5-8.5": ["2050 CHAZ SSP5-8.5 UKESM1", ...],
                }

    Returns:
        Processed disruption data for region, indexed by country ISO A3 code and scenario.
        Multi-GCM track sets are represented by their cross-model mean, with min/max
        error bar offsets in pop_at_risk_min and pop_at_risk_max respectively.
    """

    # read in data from disk
    data_by_atmosphere = []
    for atmosphere in atmospheres.keys():
        path = os.path.join(results_dir, f"power/by_storm_set/{atmosphere}/disruption/pop_affected_RP/{return_period}.gpq")
        df = gpd.read_parquet(path)

        # extend our dataframe to have countries from thresholds_by_country
        # data will be null for these rows, but it allows us to index more easily later
        missing_indicies = thresholds_by_country.index.difference(df.index)
        df = pd.concat([df, pd.DataFrame(index=missing_indicies)])

        # set countries with no calibrated value to the global default
        df["pop_at_risk"] = df[f"{default_threshold:.1f}"]
        # using the thresholds_by_country mapping, lookup population at risk for calibrated countries
        df.loc[thresholds_by_country.index, "pop_at_risk"] = lookup(
            df,
            thresholds_by_country.index,
            thresholds_by_country["threshold_ms-1"].values.astype(str)  # column labels are stringified 1 d.p. floats
        ).astype(float)
        # drop the now redundant master set of thresholds
        df = df.loc[:, ["pop_at_risk", "NAME_0", "geometry"]].sort_values("pop_at_risk", ascending=False)
        df["ATMOSPHERE"] = atmospheres[atmosphere]
        df = df.reset_index().set_index(["ISO_A3", "ATMOSPHERE"])
        data_by_atmosphere.append(df)

    data = pd.concat(data_by_atmosphere)

    # init columns for error bars
    data["pop_at_risk_min"] = 0
    data["pop_at_risk_max"] = 0

    # filter by region
    xmin, ymin, xmax, ymax = region_bbox
    data = data.cx[xmin:xmax, ymin:ymax]

    # filter out smallest countries, sorry
    mask = data.loc[:, "pop_at_risk"].groupby("ISO_A3").max() > 1E6
    notable_countries = mask[mask].index.values
    data = data[data.index.get_level_values(0).isin(notable_countries)]
    data = data.drop(columns=["NAME_0", "geometry"])

    # for each multi-GCM group, collapse individual model runs into a mean row
    # with min/max offsets for error bars
    aggregated_groups = []
    all_gcm_member_scenarios: set[str] = set()

    for group_label, member_scenarios in gcm_groups.items():
        all_gcm_member_scenarios.update(member_scenarios)

        subset = data[data.index.get_level_values("ATMOSPHERE").isin(member_scenarios)]
        groupby_country = subset.loc[:, "pop_at_risk"].groupby("ISO_A3")

        mean = groupby_country.mean()
        minimum = mean - groupby_country.min()
        minimum.name = "pop_at_risk_min"
        maximum = groupby_country.max() - mean
        maximum.name = "pop_at_risk_max"

        group_df = pd.DataFrame(
            index=pd.MultiIndex.from_product(
                [mean.index, [group_label]], names=["ISO_A3", "ATMOSPHERE"]
            )
        )
        group_df = group_df.join(mean).join(minimum).join(maximum)
        aggregated_groups.append(group_df)

    # replace individual GCM member rows with their aggregated group rows
    data = data[~data.index.get_level_values("ATMOSPHERE").isin(all_gcm_member_scenarios)]
    data = pd.concat([data] + aggregated_groups)

    data.index.names = ["Country", "Scenario"]

    return data

# estimate population connected to grid in each country
targets = pd.read_parquet(targets_path, columns=["population", "iso_a3"])
connected_pop = targets.groupby("iso_a3").sum()
connected_pop.index.name = "Country"
connected_pop = connected_pop.rename(columns={"population": "pop_at_risk"})
connected_pop["pop_at_risk_min"] = connected_pop["pop_at_risk"]
connected_pop["pop_at_risk_max"] = connected_pop["pop_at_risk"]

# assemble optimum country to threshold mapping
# calibrated thresholds, ms-1
per_country_calibrated_thresholds = pd.read_csv(country_thresholds_path).rename(columns={"iso_a3": "ISO_A3"}).set_index("ISO_A3")
# thresholds based on income, ms-1
income_threshold_map = {"L": "30.0", "LM": "30.0", "UM": "30.0", "H": "32.5"}
thresholds = pd.read_csv(wb_income_classification_path, comment="#").set_index("ISO_A3")
thresholds["threshold_ms-1"] = thresholds.INCOME_LEVEL.map(income_threshold_map)
thresholds.loc[per_country_calibrated_thresholds.index, "threshold_ms-1"] = per_country_calibrated_thresholds["threshold_ms-1"].astype(str)
# global default threshold, ms-1
default_threshold = 30.0

# Read and process data, store in `region_data`
region_data = defaultdict(dict)
for region, region_bbox in regions.items():
    for return_period in return_periods:
        print(region, return_period)
        region_data[region][return_period] = process_disruption_data(
            region,
            region_bbox,
            return_period,
            atmospheres,
            default_threshold,
            thresholds,
            results_dir,
            gcm_groups
        )

In [ ]:
def grouped_dot_plot(
    ax, df_mean, df_err_lo, df_err_hi, colours,
    width=0.85, marker="o", markersize=4,
    elinewidth=0.5, alpha=0.7,
):
    """
    df_mean: central values, index=categories, columns=series
    df_err_lo, df_err_hi: error magnitudes (already offsets, not absolute bounds)
    """
    n_categories = len(df_mean.index)
    n_series = len(df_mean.columns)

    x = np.arange(n_categories)
    edges = np.linspace(-width / 2, width / 2, n_series + 1)
    offsets = (edges[:-1] + edges[1:]) / 2

    for j, col in enumerate(df_mean.columns):
        xpos = x + offsets[j]
        y = df_mean[col].values

        yerr = np.vstack([
            df_err_lo[col].values,
            df_err_hi[col].values,
        ])

        if np.nanmin(yerr) < 0:
            print(f"Warning: negative yerr in column {col}, min value {np.nanmin(yerr)}")
        yerr = np.clip(yerr, 0, None)

        colour = colours[col] if isinstance(colours, dict) else colours[j]

        ax.errorbar(
            xpos, y, yerr=yerr,
            fmt=marker, color=colour, alpha=alpha,
            markersize=markersize, markeredgewidth=0,
            elinewidth=elinewidth, capsize=0,
            linestyle="none"
        )

    ax.set_xticks(x)
    ax.set_xticklabels(df_mean.index)
    ax.set_xlim(-0.5, n_categories - 0.5)


def summarise_trend(df_pop_at_risk):
    """
    df_pop_at_risk: DataFrame with index=Country, columns=Scenario
        (columns look like 'STORM baseline', 'STORM RCP8.5 2050', etc.)
    Returns a Series indexed by Country: sum of signs (+1 increasing, -1 decreasing)
        across the n TC models, e.g. +4 = all models increasing, -2 = net decrease.
    """
    # split each column into (model, period) — treat anything with "baseline" as
    # baseline, anything else as the future scenario for that model
    parsed = {}
    for col in df_pop_at_risk.columns:
        if "baseline" in col.lower():
            model = col.lower().replace("baseline", "").strip()
            parsed.setdefault(model, {})["baseline"] = col
        else:
            # future scenario column, e.g. "STORM RCP8.5 2050"
            model = col.split()[0].lower()
            parsed.setdefault(model, {})["future"] = col

    signs = pd.DataFrame(index=df_pop_at_risk.index)
    for model, cols in parsed.items():
        if "baseline" not in cols or "future" not in cols:
            continue  # skip incomplete pairs
        diff = df_pop_at_risk[cols["future"]] - df_pop_at_risk[cols["baseline"]]
        signs[model] = np.sign(diff)  # -1, 0, or +1

    return signs.sum(axis=1), signs


def set_between_group_grid(ax, n_categories, grid_alpha=0.15):
    # Default y grid
    ax.grid(which="minor", axis="y", alpha=grid_alpha)
    ax.grid(which="major", axis="y", alpha=grid_alpha)
    
    # Minor x ticks at the boundaries between groups (i.e. -0.5, 0.5, 1.5, ...)
    minor_locs = np.arange(-0.5, n_categories, 1)
    ax.set_xticks(minor_locs, minor=True)
    ax.grid(which="minor", axis="x", alpha=grid_alpha)
    ax.grid(which="major", axis="x", visible=False)  # avoid double gridlines if major grid was on

    # make sure minor ticks themselves aren't visible (just the gridlines)
    ax.tick_params(which="minor", axis="x", length=0)


def plot_percent_and_absolute_panel(
    return_period: int,
    region_data: dict[pd.DataFrame],
    connected_pop: pd.DataFrame,
    plot_dir: str,
    order_by: str,
    top_n: int,
) -> None:
    """
    Plot bar charts (absolute and %) of population at risk of disconnection for all regions

    Args:
        return_period: Return period of disruption event in years to lookup
        region_data: Mapping from region name to table of disruption data indexed by country and scenario
        connected_pop: Table of connected population, indexed by country
        plot_dir: Directory to save plots to
        order_by: Scenario label to order results by
    """
    spine_width = 0.5
    label_size = 5
    grid_alpha = 0.15
    legend_fontsize = 5

    f, axes = plt.subplots(
        3,
        3,
        height_ratios=[3, 2, 0.25],
        figsize=(9, 5),
    )

    for i, (region, df) in enumerate(region_data.items()):
        
        abs_ax = axes[0, i]
        rel_ax = axes[1, i]
        trend_ax = axes[2, i]
        
        n_atmospheric_states = len(df.index.get_level_values(1).unique())
    
        # Calculate percentage population at risk for later
        perc = 100 * df / connected_pop
        
        # Find fraction of connected population at risk in each case 
        df = df.unstack().sort_values([("pop_at_risk", order_by)])
        df = df.sort_index(axis=1, level=0, ascending=False)
        df = df.iloc[-top_n:]

        # Find model agreement on future trends
        trend_score, _ = summarise_trend(df.pop_at_risk)
        trend_score = trend_score.reindex(df.pop_at_risk.index)  # match plotted country order
        x = np.arange(len(trend_score))
        
        grouped_dot_plot(
            abs_ax,
            df.pop_at_risk,
            df.pop_at_risk_min,
            df.pop_at_risk_max,
            colours,
            width=0.6,
            elinewidth=0.5,
            alpha=0.7,
        )

        set_between_group_grid(abs_ax, n_categories=len(x), grid_alpha=grid_alpha)
        abs_ax.set_ylim(7E5, 1.1E8)
        abs_ax.set_yscale("log")
        abs_ax.set_xticklabels([])
        abs_ax.set_xlabel("")
        abs_ax.xaxis.set_tick_params(labelsize=label_size)
        abs_ax.yaxis.set_tick_params(labelsize=label_size)
        abs_ax.text(
            -0.2,
            0.95,
            region,
            fontsize=8,
            horizontalalignment='left',
            verticalalignment='top',
            transform=blended_transform_factory(abs_ax.transData, abs_ax.transAxes)
        )
        abs_ax.set_zorder(1)
        
        perc = perc.unstack().sort_values([("pop_at_risk", order_by)])
        perc = perc.sort_index(axis=1, level=0, ascending=False)
        # We want the absolute and percentage plots to have the same members and ordering, so index with df.index
        perc_with_abs_order = perc.loc[df.index, :]

        grouped_dot_plot(
            rel_ax,
            perc_with_abs_order.pop_at_risk,
            perc_with_abs_order.pop_at_risk_min,
            perc_with_abs_order.pop_at_risk_max,
            colours,
            width=0.6,
            elinewidth=0.5,
            alpha=0.7,
        )

        set_between_group_grid(rel_ax, n_categories=len(x), grid_alpha=grid_alpha)
        rel_ax.xaxis.set_tick_params(labelsize=label_size)
        rel_ax.yaxis.set_tick_params(labelsize=label_size)
        rel_ax.set_xlabel("", labelpad=0, fontsize=label_size)
        rel_ax.set_ylim(0, 110)

        for xi, score in zip(x, trend_score.values):
            if score == 0:
                marker, colour = "s", "grey"
            elif score > 0:
                marker, colour = "^", "firebrick"
            else:
                marker, colour = "v", "steelblue"
    
            trend_ax.scatter(
                xi, 0,
                marker=marker,
                color=colour,
                s=30 * (0.4 + abs(score) / 4),
                alpha=0.4 + 0.6 * abs(score) / 4,
                edgecolors="none",
            )
    
        trend_ax.set_xticks(x)
        trend_ax.set_xticklabels(df.pop_at_risk.index, fontsize=label_size, rotation=90)
        trend_ax.set_xlim(-0.5, len(x) - 0.5)
        trend_ax.set_yticks([])
        trend_ax.set_ylim(-1, 1)
        for spine in ["top", "right", "left"]:
            trend_ax.spines[spine].set_visible(False)
        trend_ax.spines["bottom"].set_linewidth(spine_width)
    
        # Hide tick labels on abs_ax and rel_ax now that trend_ax carries them
        abs_ax.tick_params(labelbottom=False)
        rel_ax.tick_params(labelbottom=False)

        if i == 0:
            abs_ax.set_ylabel("Population affected", labelpad=8, fontsize=label_size)
            rel_ax.set_ylabel("Population affected\n[% of connected population]", labelpad=5, fontsize=label_size)

        if i != 0:
            abs_ax.set_yticklabels([])
            rel_ax.set_yticklabels([])
        
        plt.setp(rel_ax.spines.values(), linewidth=spine_width)
        rel_ax.xaxis.set_tick_params(width=spine_width)
        rel_ax.yaxis.set_tick_params(width=spine_width)
        plt.setp(abs_ax.spines.values(), linewidth=spine_width)
        abs_ax.xaxis.set_tick_params(width=spine_width)
        abs_ax.yaxis.set_tick_params(width=spine_width)

        scenario_labels = list(df.pop_at_risk.columns)
        scenario_handles = [
            Line2D(
                [0], [0],
                marker="o", linestyle="none",
                color=colours[j] if isinstance(colours, list) else colours[label],
                alpha=0.7, markersize=4,
            )
            for j, label in enumerate(scenario_labels)
        ]
        
        trend_labels = ["Increase", "Mixed", "Decrease"]
        trend_handles = [
            Line2D([0], [0], marker="^", linestyle="none", color="firebrick", alpha=0.8, markersize=6),
            Line2D([0], [0], marker="s", linestyle="none", color="grey", alpha=0.6, markersize=5),
            Line2D([0], [0], marker="v", linestyle="none", color="steelblue", alpha=0.8, markersize=6),
        ]
        
        scenario_legend = f.legend(
            scenario_handles,
            scenario_labels,
            title="TC model / scenario",
            title_fontsize=legend_fontsize + 1,
            loc="lower center",
            bbox_to_anchor=(0.39, 0.03),
            ncol=4,
            fontsize=legend_fontsize,
            frameon=False,
        )
        
        trend_legend = f.legend(
            trend_handles,
            trend_labels,
            title="Future trend agreement",
            title_fontsize=legend_fontsize + 1,
            loc="lower center",
            bbox_to_anchor=(0.79, 0.04),
            ncol=3,
            fontsize=legend_fontsize,
            frameon=False,
        )
        
        # f.legend() replaces any previous legend on the figure by default in some
        # versions/backends — add the first legend back as an artist to keep both
        f.add_artist(scenario_legend)
    
        f.subplots_adjust(hspace=0)
        plt.subplots_adjust(left=0.08, right=0.96, top=0.96, bottom=0.18, wspace=0.05)
        
        filename = os.path.join(plot_dir, f"pop-perc-abs_rp-{return_period}.pdf")
        f.savefig(filename, dpi=254)
        plt.close(f)


# Plot bar chart of 1 in N year disruption event, by region, by country (one file per RP)
for return_period in return_periods:
    to_plot = {}
    for region in regions.keys():
        to_plot[region] = region_data[region][return_period]
    plot_percent_and_absolute_panel(return_period, to_plot, connected_pop, plot_dir, "STORM baseline", 7)